In [19]:
from neo4j import GraphDatabase
import pandas as pd
from sentence_transformers import SentenceTransformer
import numpy as np
import tqdm

In [20]:
intention_categories = ["player_basic_info", "player_season_stats", "player_gw_stats", 
                        "fixture_details", "team_fixtures", "team_players", "top_players_position","compare_players","gameweek_summary","player_vs_opponent"
]

intention_map = {
    "player_basic_info": """
MATCH (p:Player {player_name: $player})
OPTIONAL MATCH (p)-[:PLAYS_AS]->(pos:Position)
OPTIONAL MATCH (p)-[:PLAYS_FOR {season: $season}]->(t:Team)
RETURN p, pos, t;""",
    "player_season_stats": """
MATCH (p:Player {player_name: $player})-[r:PLAYED_IN]->(f:Fixture)
MATCH (f)<-[:HAS_FIXTURE]-(g:Gameweek)<-[:HAS_GW]-(s:Season {season_name: $season})
RETURN p.player_name AS player,
       SUM(r.total_points) AS total_points,
       SUM(r.goals_scored) AS goals,
       SUM(r.assists) AS assists,
       SUM(r.minutes) AS minutes,
       AVG(r.form) AS avg_form;
""",
    "player_gw_stats":"""
MATCH (p:Player {player_name: $player})-[r:PLAYED_IN]->(f:Fixture)
MATCH (g:Gameweek {season: $season, GW_number: $gw})-[:HAS_FIXTURE]->(f)
RETURN p, r, f;
""",
    "fixture_details":"""
MATCH (p:Player {player_name: $player})-[r:PLAYED_IN]->(f:Fixture)
MATCH (g:Gameweek {season: $season, GW_number: $gw})-[:HAS_FIXTURE]->(f)
RETURN p, r, f;
""",
    "team_fixtures":"""
MATCH (t:Team {name: $team})
MATCH (f:Fixture)-[:HAS_HOME_TEAM|HAS_AWAY_TEAM]->(t)
RETURN f ORDER BY f.kickoff_time;
""",
    "team_players":"""
MATCH (t:Team {name: $team})
MATCH (p:Player)-[:PLAYS_FOR {season: $season}]->(t)
RETURN p;
""",
    "top_players_position":"""
MATCH (p:Player)-[:PLAYS_AS]->(:Position {name: $position})
MATCH (p)-[r:PLAYED_IN]->(f:Fixture)
MATCH (f)<-[:HAS_FIXTURE]-(g:Gameweek)<-[:HAS_GW]-(s:Season {season_name: $season})
RETURN p.player_name AS player, SUM(r.total_points) AS points
ORDER BY points DESC LIMIT $limit;
""",
    "compare_players":"""
MATCH (p1:Player {player_name: $player1})-[r1:PLAYED_IN]->(f1:Fixture)
MATCH (p2:Player {player_name: $player2})-[r2:PLAYED_IN]->(f2:Fixture)
RETURN p1.player_name AS player1, SUM(r1.total_points) AS p1_points,
       p2.player_name AS player2, SUM(r2.total_points) AS p2_points;
""",
    "gameweek_summary":"""
MATCH (g:Gameweek {season: $season, GW_number: $gw})-[:HAS_FIXTURE]->(f)
MATCH (p:Player)-[r:PLAYED_IN]->(f)
RETURN g, f, p, r
ORDER BY f.fixture_number;
""",
    "player_vs_opponent":"""
MATCH (p:Player {player_name: $player})-[r:PLAYED_IN]->(f:Fixture)
MATCH (p)-[:PLAYED_AGAINST]->(opp:Team {name: $opponent})
RETURN p.player_name AS player, opp.name AS opponent,
       SUM(r.goals_scored) AS goals,
       SUM(r.assists) AS assists,
       SUM(r.total_points) AS points;
"""
}

In [ ]:
from dotenv import load_dotenv
from openai import OpenAI
import os
load_dotenv(override=True)

# Load API key from environment variable for security
api_key = os.getenv("OPEN_ROUTER_KEY")
client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=api_key,
)

def call_openAI(prompt: str) -> str:
    completion = client.chat.completions.create(
        model="gpt-3.5-turbo",
        messages=[
            {
                "role": "user",
                "content": [
                    {
                        "type": "text",
                        "text": prompt
                    }
                ]
            }
        ],
        max_tokens=1000  # Limit the response length to reduce cost
    )
    return completion.choices[0].message.content
def call_open_router(prompt: str) -> str:
    completion = client.chat.completions.create(
        extra_body={},
        model="meta-llama/llama-3.3-70b-instruct:free",
        messages=[
            {
                "role": "user",
                "content": [
                    {
                        "type": "text",
                        "text": prompt
                    }
                ]
            }
        ],
        max_tokens=1000  # Limit the response length to reduce cost
    )
    return completion.choices[0].message.content


In [22]:
# We are going to use LLM-based classification
def classify_intent(user_input):
    # TODO: This should return the Cypher Queries (descriptions) associated with each intention Or the Retrieval methods to use.

    prompt = f"""
You are an intent classifier for a Fantasy Premier League (FPL) knowledge graph system.

Your task:
Given a user query, classify it into EXACTLY one of the following categories:
{', '.join(intention_categories)}

Category definitions (important):
- player_basic_info: Asking who a player is, their position, or their team.
- player_season_stats: Asking about a player's overall seasonal performance.
- player_gw_stats: Asking about a player's performance in a specific gameweek.
- fixture_details: Asking about a specific match, its teams, or players in it.
- team_fixtures: Asking about a team's upcoming or past fixtures.
- team_players: Asking which players belong to a team.
- top_players_position: Asking for ranking or best players in a position/season.
- compare_players: Comparing two players statistically.
- gameweek_summary: Asking about all fixtures or events in a specific GW.
- player_vs_opponent: Asking how a player performed against a specific team.

Examples:
User Input: "Show me Haaland's stats last season."
Category: player_season_stats

User Input: "How did Salah do in GW 5?"
Category: player_gw_stats

User Input: "Who plays for Arsenal this season?"
Category: team_players

User Input: "Which fixtures does Liverpool have next month?"
Category: team_fixtures

User Input: "Tell me which defender scored the most points last year."
Category: top_players_position

User Input: "Compare Son and Rashford this season."
Category: compare_players

User Input: "What happened in gameweek 10?"
Category: gameweek_summary

User Input: "How does Kane perform against Chelsea?"
Category: player_vs_opponent

User Input: "Who is Trent Alexander-Arnold?"
Category: player_basic_info

Now classify the user's input below.
Return ONLY the category name from the list above.

User Input: "{user_input}"
Category:
"""
    category = call_open_router(prompt).strip()
    if category not in intention_categories:
        print("Warning: LLM returned an unexpected category.")
        print(f"LLM Output: {category}")
        category = "Unknown"
    return category, intention_map.get(category)

In [23]:
from neo4j import GraphDatabase

# Initialize Neo4j connection for entity grounding
config = {}
with open("config.txt", "r") as f:
    for line in f:
        key, value = line.strip().split("=", 1)
        config[key] = value

neo4j_driver = GraphDatabase.driver(config["URI"], auth=(config["USERNAME"], config["PASSWORD"]))

def get_kg_entities():
    """
    Retrieve all entity values from the knowledge graph to ground entity extraction.
    Returns dictionaries of players, teams, positions, and seasons.
    """
    with neo4j_driver.session() as session:
        # Get all players
        players = session.run("MATCH (p:Player) RETURN p.player_name as name").data()
        player_names = [p['name'] for p in players if p['name']]
        
        # Get all teams
        teams = session.run("MATCH (t:Team) RETURN t.name as name").data()
        team_names = [t['name'] for t in teams if t['name']]
        
        # Get all positions
        positions = session.run("MATCH (pos:Position) RETURN pos.name as name").data()
        position_names = [pos['name'] for pos in positions if pos['name']]
        
        # Get all seasons
        seasons = session.run("MATCH (s:Season) RETURN s.season_name as name").data()
        season_names = [s['name'] for s in seasons if s['name']]
        
        # Get gameweek range
        gameweeks = session.run("MATCH (g:Gameweek) RETURN DISTINCT g.GW_number as gw ORDER BY gw").data()
        gw_numbers = [gw['gw'] for gw in gameweeks if gw['gw']]
        
    return {
        'players': player_names,
        'teams': team_names,
        'positions': position_names,
        'seasons': season_names,
        'gameweeks': gw_numbers
    }

# Cache KG entities for faster lookups
kg_entities = get_kg_entities()
print(f"Loaded {len(kg_entities['players'])} players, {len(kg_entities['teams'])} teams, "
      f"{len(kg_entities['positions'])} positions, {len(kg_entities['seasons'])} seasons")


Loaded 1323 players, 31 teams, 4 positions, 5 seasons


In [24]:
import re
from difflib import get_close_matches

def extract_entities(user_input):
    """
    Extract and ground entities from user input using the knowledge graph.
    Returns a structured dictionary with entity types and values validated against the KG.
    """
    
    # Step 1: Use LLM to identify potential entities and their types
    prompt = f"""Extract entities from the following fantasy football query. For each entity, identify its type.
Return the result in this exact format: EntityType: value1, value2
Available entity types: Player, Team, Position, Season, Gameweek, Statistic, TimeReference
Do not include any explanations or additional text.

Examples:
User Input: "Who is the top scoring midfielder this season?"
Player: 
Team: 
Position: midfielder
Season: this season
Gameweek: 
Statistic: top scoring
TimeReference: this season

User Input: "Find me a West Ham midfielder that scored the most points last season"
Player: 
Team: West Ham
Position: midfielder
Season: last season
Gameweek: 
Statistic: most points
TimeReference: last season

User Input: "How many goals did Salah score in gameweek 5?"
Player: Salah
Team: 
Position: 
Season: 
Gameweek: 5
Statistic: goals
TimeReference: gameweek 5

User Input: "{user_input}"
Player: 
Team: 
Position: 
Season: 
Gameweek: 
Statistic: 
TimeReference: 
"""
    
    llm_response = call_open_router(prompt).strip()
    print("llm_response: ",llm_response)
    # Step 2: Parse LLM response
    extracted = {
        'players': [],
        'teams': [],
        'positions': [],
        'seasons': [],
        'gameweeks': [],
        'statistics': [],
        'time_references': []
    }
    
    lines = llm_response.split('\n')
    for line in lines:
        if ':' in line:
            entity_type, values = line.split(':', 1)
            entity_type = entity_type.strip().lower()
            values = values.strip()
            
            if values and values.lower() not in ['none', 'n/a', '']:
                value_list = [v.strip() for v in values.split(',') if v.strip()]
                
                if 'player' in entity_type:
                    extracted['players'].extend(value_list)
                elif 'team' in entity_type:
                    extracted['teams'].extend(value_list)
                elif 'position' in entity_type:
                    extracted['positions'].extend(value_list)
                elif 'season' in entity_type:
                    extracted['seasons'].extend(value_list)
                elif 'gameweek' in entity_type:
                    extracted['gameweeks'].extend(value_list)
                elif 'statistic' in entity_type:
                    extracted['statistics'].extend(value_list)
                elif 'time' in entity_type:
                    extracted['time_references'].extend(value_list)
    
    # Step 3: Ground entities against the knowledge graph
    grounded_entities = {
        'players': [],
        'teams': [],
        'positions': [],
        'seasons': [],
        'gameweeks': [],
        'statistics': [],
        'time_references': extracted['time_references']
    }
    
    # Ground players
    for player in extracted['players']:
        matches = get_close_matches(player, kg_entities['players'], n=3, cutoff=0.2)
        if matches:
            grounded_entities['players'].append({
                'original': player,
                'grounded': matches[0],
                'alternatives': matches[1:] if len(matches) > 1 else []
            })
    
    # Ground teams
    for team in extracted['teams']:
        matches = get_close_matches(team, kg_entities['teams'], n=3, cutoff=0.2)
        if matches:
            grounded_entities['teams'].append({
                'original': team,
                'grounded': matches[0],
                'alternatives': matches[1:] if len(matches) > 1 else []
            })
    
    # Ground positions (normalize to KG format)
    position_mapping = {
        'goalkeeper': 'GK',
        'gk': 'GK',
        'defender': 'DEF',
        'def': 'DEF',
        'midfielder': 'MID',
        'mid': 'MID',
        'forward': 'FWD',
        'fwd': 'FWD',
        'striker': 'FWD',
        'attacker': 'FWD'
    }
    
    for position in extracted['positions']:
        position_lower = position.lower()
        if position_lower in position_mapping:
            mapped_pos = position_mapping[position_lower]
            if mapped_pos in kg_entities['positions']:
                grounded_entities['positions'].append({
                    'original': position,
                    'grounded': mapped_pos
                })
        else:
            matches = get_close_matches(position, kg_entities['positions'], n=1, cutoff=0.2)
            if matches:
                grounded_entities['positions'].append({
                    'original': position,
                    'grounded': matches[0]
                })
    
    # Ground seasons
    for season in extracted['seasons']:
        # Handle relative references
        if 'this' in season.lower() or 'current' in season.lower():
            latest_season = max(kg_entities['seasons']) if kg_entities['seasons'] else None
            if latest_season:
                grounded_entities['seasons'].append({
                    'original': season,
                    'grounded': latest_season,
                    'is_relative': True
                })
        elif 'last' in season.lower() or 'previous' in season.lower():
            sorted_seasons = sorted(kg_entities['seasons'], reverse=True)
            if len(sorted_seasons) > 1:
                grounded_entities['seasons'].append({
                    'original': season,
                    'grounded': sorted_seasons[1],
                    'is_relative': True
                })
        else:
            matches = get_close_matches(season, kg_entities['seasons'], n=1, cutoff=0.2)
            if matches:
                grounded_entities['seasons'].append({
                    'original': season,
                    'grounded': matches[0],
                    'is_relative': False
                })
    
    # Extract gameweek numbers
    for gw in extracted['gameweeks']:
        # Extract numeric value
        gw_match = re.search(r'\d+', gw)
        if gw_match:
            gw_num = int(gw_match.group())
            if gw_num in kg_entities['gameweeks']:
                grounded_entities['gameweeks'].append({
                    'original': gw,
                    'grounded': gw_num
                })
    
    # Keep statistics as-is (these are performance metrics)
    grounded_entities['statistics'] = extracted['statistics']
    
    return extracted, grounded_entities


In [25]:
def populate_query(query, entities):
    """
    Populate a Cypher query template with extracted and grounded entities.
    
    Args:
        query (str): Cypher query template with parameter placeholders (e.g., $player, $season)
        entities (dict): Dictionary of grounded entities from extract_entities function
        
    Returns:
        tuple: (populated_query, parameters_dict)
            - populated_query: The original query (for Neo4j driver execution)
            - parameters_dict: Dictionary of parameters to pass to Neo4j
    """
    
    # Extract all parameter names from the query using regex
    # Matches $parameter_name patterns
    param_pattern = r'\$(\w+)'
    required_params = set(re.findall(param_pattern, query))
    
    # Initialize parameters dictionary
    parameters = {}
    
    # Mapping of parameter names to entity types
    param_to_entity_map = {
        'player': 'players',
        'player1': 'players',
        'player2': 'players',
        'team': 'teams',
        'opponent': 'teams',
        'position': 'positions',
        'season': 'seasons',
        'gw': 'gameweeks',
        'limit': 'statistics'  # Special case for LIMIT clauses
    }
    
    # Populate parameters based on extracted entities
    for param in required_params:
        entity_type = param_to_entity_map.get(param)
        
        if entity_type and entity_type in entities:
            entity_list = entities[entity_type]
            
            if entity_list:
                if param == 'player1' and len(entity_list) >= 1:
                    # For player comparisons, use first player
                    parameters[param] = entity_list[0]['grounded']
                elif param == 'player2' and len(entity_list) >= 2:
                    # For player comparisons, use second player
                    parameters[param] = entity_list[1]['grounded']
                elif param in ['player', 'team', 'opponent', 'position', 'season']:
                    # Use the grounded value from the first match
                    parameters[param] = entity_list[0]['grounded']
                elif param == 'gw':
                    # Gameweek should be an integer
                    parameters[param] = entity_list[0]['grounded']
                elif param == 'limit':
                    # Extract number from statistics if present
                    # Default to 10 if not specified
                    limit_value = 10
                    if entities.get('statistics'):
                        for stat in entities['statistics']:
                            # Try to extract number from phrases like "top 5", "best 10"
                            num_match = re.search(r'\d+', stat)
                            if num_match:
                                limit_value = int(num_match.group())
                                break
                    parameters[param] = limit_value
    
    # Check for missing required parameters (non-OPTIONAL matches)
    # This is a simple heuristic - you may want to make this more sophisticated
    missing_params = []
    for param in required_params:
        if param not in parameters:
            # Check if the parameter is in an OPTIONAL MATCH clause
            # If not, it's required
            optional_pattern = rf'OPTIONAL\s+MATCH.*\${param}\b'
            if not re.search(optional_pattern, query, re.IGNORECASE | re.DOTALL):
                missing_params.append(param)
    
    if missing_params:
        print(f"Warning: Missing required parameters: {missing_params}")
        print(f"Available entities: {list(entities.keys())}")
        return None, None
    
    return parameters

In [26]:
df = pd.read_csv('fpl_graph_final.csv')

In [27]:
config = {}
with open("config.txt", "r") as f:
    for line in f:
        key, value = line.strip().split("=", 1)
        config[key] = value

driver = GraphDatabase.driver(config["URI"], auth=(config["USERNAME"], config["PASSWORD"]))

with driver.session() as session:
    result = session.run(
        """
        MATCH (p:Player)-[r:PLAYED_IN]->(f:Fixture)
        OPTIONAL MATCH (p)-[:PLAYS_AS]->(pos:Position)
        OPTIONAL MATCH (p)-[:PLAYS_FOR]->(t:Team)
        RETURN
            p.code AS code,
            p.player_name    AS name,
            coalesce(pos.name, '') AS position,
            coalesce(head(collect(DISTINCT t.name)), '') AS team,
            SUM(r.goals_scored)      AS sum_goals,
            SUM(r.assists)           AS sum_assists,
            SUM(r.total_points)      AS sum_points,
            SUM(r.minutes)           AS sum_minutes,
            SUM(r.clean_sheets)      AS sum_clean_sheets,
            SUM(r.goals_conceded)    AS sum_goals_conceded,
            SUM(r.bps)               AS sum_bps,
            AVG(r.influence)         AS avg_influence,
            AVG(r.creativity)        AS avg_creativity,
            AVG(r.threat)            AS avg_threat,
            AVG(r.ict_index)         AS avg_ict_index
        """
    )

    df = pd.DataFrame(result.data())

driver.close()

print("Rows fetched from Neo4j:", len(df))

Rows fetched from Neo4j: 1340


In [28]:
numeric_cols = [
    "sum_goals",
    "sum_assists",
    "sum_points",
    "sum_minutes",
    "sum_clean_sheets",
    "sum_goals_conceded",
    "sum_bps",
    "avg_influence",
    "avg_creativity",
    "avg_threat",
    "avg_ict_index"
]
# Ensure numeric dtypes
df[numeric_cols] = df[numeric_cols].astype(float)

# Optional: standardize features (recommended)
means = df[numeric_cols].mean()
stds = df[numeric_cols].replace(0, np.nan).std().replace(0, np.nan)

df_norm = (df[numeric_cols] - means) / stds

# Fill NaNs (if any) with 0 after normalization
df_norm = df_norm.fillna(0.0)

# Build list-of-floats vectors
df["embedding_numeric"] = df_norm.apply(lambda row: row.values.astype(float).tolist(), axis=1)

print("Example numeric embedding for first player:")
print(df["embedding_numeric"].iloc[0])
print("Length:", len(df["embedding_numeric"].iloc[0]))

Example numeric embedding for first player:
[2.1602666935802923, 1.7600302103096142, 2.546450571864651, 2.224508496940185, 1.7047705477021047, 2.572937317279771, 1.5660559959908202, 0.25571265420621725, 0.2413089612263, 0.9532532172748948, 0.5950020240344032]
Length: 11


In [29]:
# Build a textual description per player for the text model
def build_player_description(row):
    return (
        f"Player: {row['name']}, "
        f"Position: {row['position']}, "
        f"Team: {row['team']}, "
        f"Goals: {row['sum_goals']}, "
        f"Assists: {row['sum_assists']}, "
        f"Total points: {row['sum_points']}, "
        f"Minutes played: {row['sum_minutes']}, "
        f"Clean sheets: {row['sum_clean_sheets']}, "
        f"Influence: {row['avg_influence']:.1f}, "
        f"Creativity: {row['avg_creativity']:.1f}, "
        f"Threat: {row['avg_threat']:.1f}, "
        f"ICT index: {row['avg_ict_index']:.1f}."
    )

In [30]:
df["description"] = df.apply(build_player_description, axis=1)

print("Sample description:")
print(df["description"].iloc[0])

Sample description:
Player: Theo Walcott, Position: MID, Team: Arsenal, Goals: 126.0, Assists: 77.0, Total points: 2212.0, Minutes played: 43274.0, Clean sheets: 133.0, Influence: 6.9, Creativity: 5.0, Threat: 10.8, ICT index: 2.3.


In [31]:
# Load a HuggingFace embedding model (you can change this later to compare models)
model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
model_v2 = SentenceTransformer("sentence-transformers/all-mpnet-base-v2")


# Compute embeddings for all players at once
text_embeddings = model.encode(
    df["description"].tolist(),
    batch_size=64,
    show_progress_bar=True,
    convert_to_numpy=True
)

text_embeddings_v2 = model_v2.encode(
    df["description"].tolist(),
    batch_size=32,
    show_progress_bar=True,
    convert_to_numpy=True
)

# Attach to dataframe as list-of-floats
df["embedding_text_l6_v2"] = [vec.astype(float).tolist() for vec in text_embeddings]
df["embedding_text_mpnet_v2"] = [vec.astype(float).tolist() for vec in text_embeddings_v2]

print("Example text embedding for first player:")
print(df["embedding_text_l6_v2"].iloc[0][:10], "...")  # first 10 dims
print("Length:", len(df["embedding_text_l6_v2"].iloc[0]))

print("Example mpnet text embedding for first player:")
print(df["embedding_text_mpnet_v2"].iloc[0][:10], "...")  # first 10 dims
print("Length:", len(df["embedding_text_mpnet_v2"].iloc[0]))

Batches: 100%|██████████| 42/42 [04:38<00:00,  6.62s/it]

Example text embedding for first player:
[-0.012594807893037796, 0.0013771603116765618, -0.025190338492393494, -0.022800235077738762, 0.07064928859472275, 0.14169958233833313, 0.08784733712673187, 0.06699337810277939, 0.047621846199035645, 0.0016818670555949211] ...
Length: 384
Example mpnet text embedding for first player:
[-0.06834255903959274, -0.020946795120835304, 0.0025531984865665436, 0.054962776601314545, -0.01291707158088684, -0.03988290950655937, -0.021269632503390312, -7.386863580904901e-05, 0.0007107221754267812, 0.0171197559684515] ...
Length: 768


In [32]:
driver = GraphDatabase.driver(config["URI"], auth=(config["USERNAME"], config["PASSWORD"]))

def write_embeddings(tx, code, emb_numeric, emb_text):
    tx.run(
        """
        MATCH (p:Player {code: $code})
        SET p.embedding_numeric = $embedding_numeric,
            p.embedding_text_l6_v2 = $embedding_text_l6_v2
        """,
        code=code,
        embedding_numeric=emb_numeric,
        embedding_text_l6_v2=emb_text
    )

def write_embeddings_v2(tx, code, emb_text_v2):
    tx.run(
        """
        MATCH (p:Player {code: $code})
        SET p.embedding_text_mpnet_v2 = $embedding_text_mpnet_v2
        """,
        code=code,
        embedding_text_mpnet_v2=emb_text_v2
    )

with driver.session() as session:
    # Iterate through players
    for _, row in tqdm.tqdm(df.iterrows(), total=len(df)):
        session.execute_write(
            write_embeddings,
            row["code"],
            row["embedding_numeric"],
            row["embedding_text_l6_v2"]
        )
        session.execute_write(
            write_embeddings_v2,
            row["code"],
            row["embedding_text_mpnet_v2"]
        )

print("Embeddings successfully written to Neo4j!")
driver.close()

100%|██████████| 1340/1340 [00:42<00:00, 31.59it/s]

Embeddings successfully written to Neo4j!


In [33]:
driver = GraphDatabase.driver(config["URI"], auth=(config["USERNAME"], config["PASSWORD"]))
def create_indexes(tx):
    # Numeric embedding index (dimension = 11)
    tx.run("""
        CREATE VECTOR INDEX playerEmbeddingNumericIndex
        IF NOT EXISTS
        FOR (p:Player) ON (p.embedding_numeric)
        OPTIONS {
            indexConfig: {
                `vector.dimensions`: 11,
                `vector.similarity_function`: "cosine"
            }
        };
    """)

    # Text embedding index (dimension = 384)
    tx.run("""
        CREATE VECTOR INDEX playerEmbeddingTextIndexL6V2
        IF NOT EXISTS
        FOR (p:Player) ON (p.embedding_text_l6_v2)
        OPTIONS {
            indexConfig: {
                `vector.dimensions`: 384,
                `vector.similarity_function`: "cosine"
            }
        };
    """)

    tx.run("""
        CREATE VECTOR INDEX playerEmbeddingTextIndexMpnetV2
        IF NOT EXISTS
        FOR (p:Player) ON (p.embedding_text_mpnet_v2)
        OPTIONS {
            indexConfig: {
                `vector.dimensions`: 768,
                `vector.similarity_function`: "cosine"
            }
        };
    """)

with driver.session() as session:
    session.execute_write(create_indexes)

driver.close()

print("Vector indexes created successfully!")

Vector indexes created successfully!


In [34]:
def get_similar_players(player_name, mode="text", top_k=5):
    if mode == "text":
        embedding_prop = "embedding_text_l6_v2"
        index_name = "playerEmbeddingTextIndexL6V2"
    elif mode == "numeric":
        embedding_prop = "embedding_numeric"
        index_name = "playerEmbeddingNumericIndex"
    elif mode == "text_v2":
        embedding_prop = "embedding_text_mpnet_v2"
        index_name = "playerEmbeddingTextIndexMpnetV2"
    else:
        raise ValueError("mode must be 'text', 'text_v2', or 'numeric'")

    query = f"""
    MATCH (p:Player {{player_name: $name}})
    WITH p.{embedding_prop} AS query_vec
    CALL db.index.vector.queryNodes('{index_name}', $top_k, query_vec)
    YIELD node, score
    RETURN node.player_name AS similar_player, score
    ORDER BY score DESC
    """

    with driver.session() as session:
        results = session.run(query, name=player_name, top_k=top_k)
        return [(r["similar_player"], r["score"]) for r in results]


In [35]:
def normalize_baseline_result(result_list):
    """
    Converts Neo4j baseline output into a unified format.
    Handles nodes (Player, Team, Position, Fixture), relationships, and aggregated stats.
    """
    unified = []

    for record in result_list:
        item = {
            "type": "structured",
            "players": [],
            "teams": [],
            "positions": [],
            "fixtures": [],
            "relationships": [],
            "aggregated_stats": {}
        }

        for key, value in record.items():

            # Player node
            if key == "p" or key == "player" or key.startswith("player"):
                if isinstance(value, dict):  # from Neo4j node
                    item["players"].append({
                        "player_name": value.get("player_name", "Unknown"),
                        "player_code": value.get("code"),
                        "other_props": {k: v for k, v in value.items() if k not in ["player_name", "code"]}
                    })
                else:  # aggregated name string
                    item["players"].append({"player_name": value})

            # Team node
            elif key == "t":
                item["teams"].append({
                    "team_name": value.get("name", "Unknown") if isinstance(value, dict) else str(value)
                })

            # Position node
            elif key == "pos":
                item["positions"].append({
                    "position_name": value.get("name", "Unknown") if isinstance(value, dict) else str(value)
                })

            # Fixture node
            elif key == "f":
                if isinstance(value, dict):
                    item["fixtures"].append({
                        "fixture_number": value.get("fixture_number"),
                        "kickoff_time": value.get("kickoff_time"),
                        "season": value.get("season"),
                        "other_props": {k: v for k, v in value.items() if k not in ["fixture_number", "kickoff_time", "season"]}
                    })
                else:
                    item["fixtures"].append({"fixture": value})

            # Relationships (tuple format)
            elif key == "r" and isinstance(value, tuple) and len(value) == 3:
                start, rel, end = value
                item["relationships"].append({
                    "type": rel if isinstance(rel, str) else getattr(rel, "type", "unknown"),
                    "from": getattr(start, "get", lambda x, d=None: d)("player_name", getattr(start, "name", "Unknown")),
                    "to": getattr(end, "get", lambda x, d=None: d)("player_name", getattr(end, "name", "Unknown")),
                    "props": rel.items() if hasattr(rel, "items") else {}
                })

            # Aggregated stats
            elif key in ["assists", "minutes", "avg_form", "total_points", "goals", "points", "player1_points", "player2_points"]:
                item["aggregated_stats"][key] = value

            # Gameweek / Season nodes
            elif key == "g":
                if isinstance(value, dict):
                    item["gameweek"] = value
                else:
                    item["gameweek"] = {"value": value}

            # Player comparison
            elif key in ["player1", "player2"]:
                item["players"].append({"player_name": value})

            else:
                # fallback: store in aggregated_stats
                item["aggregated_stats"][key] = value

        unified.append(item)

    return unified


In [63]:
def get_context(question):
    # 1️⃣ Extract structured query info
    category, query = classify_intent(question)
    _, grounded_entities = extract_entities(question)
    query_params = populate_query(query, grounded_entities)
    neo4j_driver = GraphDatabase.driver(config["URI"], auth=(config["USERNAME"], config["PASSWORD"]))

    # 2️⃣ Run baseline Cypher query
    with neo4j_driver.session() as session:
        result1 = session.run(query, query_params)
        baseline = result1.data()
    print("Baseline result:", baseline)
    print("Grounded entities:", grounded_entities)
    print("Query params:", query_params)
    print("Cypher query:", query)
    normalized_baseline = normalize_baseline_result(baseline)

    # 3️⃣ Prepare player names for embedding-based retrieval
    players = grounded_entities.get("Players", [])
    players_name = [player['grounded'] for player in players]
    print(players_name)
    # 4️⃣ Get embedding-based contexts
    embeddings_context = []
    for player_name in players_name:
        context = get_similar_players(player_name, mode="text_v2")
        embeddings_context.extend(context)  # flatten

    # 5️⃣ Combine results
    unified_context = []

    # Add embedding-based results, avoid duplicates
    for player_name, score in embeddings_context:
        unified_context.append({
            "type": "semantic",
            "player_name": player_name,
            "similarity_score": score
           })
    unified_context.extend(normalized_baseline)

    return unified_context,normalized_baseline


In [37]:
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.messages import HumanMessage, SystemMessage

# Load environment variables
load_dotenv(override=True)

# Verify keys are loaded
print("OpenAI Key loaded:", bool(os.getenv("OPENAI_KEY")))
print("OpenRouter Key loaded:", bool(os.getenv("OPEN_ROUTER_KEY")))
print("Google API Key loaded:", bool(os.getenv("GEMINI_KEY")))

OPENAI_KEY = KEY = os.getenv("OPENAI_KEY")
OPENROUTER_KEY = KEY = os.getenv("OPEN_ROUTER_KEY")
GEMINI_KEY = KEY = os.getenv("GEMINI_KEY")

OpenAI Key loaded: True
OpenRouter Key loaded: True
Google API Key loaded: True


In [38]:
# Persona: Define the assistant's role with comprehensive guidelines
SystemPrompt = """You are an expert Fantasy Premier League (FPL) assistant with deep knowledge of player statistics, team performance, fixtures, and FPL strategy. You provide accurate, data-driven insights to help FPL managers make informed decisions.

Guidelines for your responses:
- Carefully analyze the context provided to extract relevant information
- Answer questions accurately using ONLY the information available in the context
- If the context doesn't contain enough information to answer the question, clearly state that
- Provide specific statistics, player names, team names, and gameweek data when available in the context
- Be concise but comprehensive in your answers
- Use FPL terminology correctly (e.g., GW for gameweek, xG for expected goals, xA for expected assists, ICT index, bonus points, BPS, etc.)
- When discussing player performance, include relevant metrics like points scored, goals, assists, clean sheets, and bonus points if available
- For team-related questions, reference fixtures, form, and statistics from the context
- Maintain an enthusiastic and knowledgeable tone about Fantasy Premier League
- Base all recommendations and insights strictly on the provided context to avoid hallucinations"""

In [39]:
import time

def openai_generate(context: str, question: str) -> dict:
    """
    Generate response using OpenAI model.
    
    Args:
        context: The retrieved knowledge graph information (nodes, relationships, data)
        question: The user's question to answer
        
    Returns:
        Dictionary containing:
        - response: String response from the model
        - metrics: Dictionary with response_time, token_usage, and cost
    """
    openai_llm = ChatOpenAI(
        model="gpt-5.1",
        temperature=0.7,
        api_key=OPENAI_KEY,
        max_tokens=1000
    )
    
    # Structure: Persona (SystemMessage) + Context + Task (HumanMessage)
    messages = [
        SystemMessage(content=SystemPrompt),
        HumanMessage(content=f"""Context:
{context}

Task:
Answer the following question using ONLY the information provided in the context above. If the context doesn't contain enough information to answer the question, clearly state that. Be specific and cite relevant statistics, player names, or data from the context.

Question: {question}""")
    ]
    
    # Measure response time
    start_time = time.time()
    response = openai_llm.invoke(messages)
    end_time = time.time()
    
    # Extract token usage
    prompt_tokens = response.response_metadata.get('token_usage', {}).get('prompt_tokens', 0)
    completion_tokens = response.response_metadata.get('token_usage', {}).get('completion_tokens', 0)
    total_tokens = response.response_metadata.get('token_usage', {}).get('total_tokens', 0)
    
    # Calculate cost (GPT-5.1 pricing: $1.25 per 1M prompt tokens, $10 per 1M completion tokens)
    cost = (prompt_tokens / 1000000 * 1.25) + (completion_tokens / 1000000 * 10)
    
    return {
        "response": response.content,
        "metrics": {
            "response_time": round(end_time - start_time, 2),
            "token_usage": {
                "prompt_tokens": prompt_tokens,
                "completion_tokens": completion_tokens,
                "total_tokens": total_tokens
            },
            "cost": round(cost, 6)
        }
    }

In [40]:
def openrouter_generate(context: str, question: str) -> dict:
    """
    Generate response using OpenRouter model.
    
    Args:
        context: The retrieved knowledge graph information (nodes, relationships, data)
        question: The user's question to answer
        
    Returns:
        Dictionary containing:
        - response: String response from the model
        - metrics: Dictionary with response_time, token_usage, and cost
    """
    openrouter_llm = ChatOpenAI(
        model="meta-llama/llama-3.3-70b-instruct:free",
        temperature=0.7,
        api_key=OPENROUTER_KEY,
        base_url="https://openrouter.ai/api/v1",
        max_tokens=1000,
        default_headers={
            "HTTP-Referer": "http://localhost",
            "X-Title": "LangChain Template"
        }
    )
    
    # Structure: Persona (SystemMessage) + Context + Task (HumanMessage)
    messages = [
        SystemMessage(content=SystemPrompt),
        HumanMessage(content=f"""Context:
{context}

Task:
Answer the following question using ONLY the information provided in the context above. If the context doesn't contain enough information to answer the question, clearly state that. Be specific and cite relevant statistics, player names, or data from the context.

Question: {question}""")
    ]
    
    # Measure response time
    start_time = time.time()
    response = openrouter_llm.invoke(messages)
    end_time = time.time()
    
    # Extract token usage
    prompt_tokens = response.response_metadata.get('token_usage', {}).get('prompt_tokens', 0)
    completion_tokens = response.response_metadata.get('token_usage', {}).get('completion_tokens', 0)
    total_tokens = response.response_metadata.get('token_usage', {}).get('total_tokens', 0)
    
    # Cost for free model is $0
    cost = 0.0
    
    return {
        "response": response.content,
        "metrics": {
            "response_time": round(end_time - start_time, 2),
            "token_usage": {
                "prompt_tokens": prompt_tokens,
                "completion_tokens": completion_tokens,
                "total_tokens": total_tokens
            },
            "cost": round(cost, 6)
        }
    }

In [41]:
def gemini_generate(context: str, question: str) -> dict:
    """
    Generate response using Gemini model.
    
    Args:
        context: The retrieved knowledge graph information (nodes, relationships, data)
        question: The user's question to answer
        
    Returns:
        Dictionary containing:
        - response: String response from the model
        - metrics: Dictionary with response_time, token_usage, and cost
    """
    gemini_llm = ChatGoogleGenerativeAI(
        model="gemini-2.5-flash",
        temperature=0.7,
        google_api_key=GEMINI_KEY,
        max_tokens=1000
    )
    
    # Structure: Persona (SystemMessage) + Context + Task (HumanMessage)
    messages = [
        SystemMessage(content=SystemPrompt),
        HumanMessage(content=f"""Context:
{context}

Task:
Answer the following question using ONLY the information provided in the context above. If the context doesn't contain enough information to answer the question, clearly state that. Be specific and cite relevant statistics, player names, or data from the context.

Question: {question}""")
    ]
    
    # Measure response time
    start_time = time.time()
    response = gemini_llm.invoke(messages)
    end_time = time.time()
    
    # Extract token usage
    usage_metadata = response.usage_metadata
    prompt_tokens = usage_metadata.get('input_tokens', 0)
    completion_tokens = usage_metadata.get('output_tokens', 0)
    total_tokens = usage_metadata.get('total_tokens', 0)
    
    # Calculate cost (Gemini 2.5 Flash pricing: $0.03 per 1M input tokens, $2.50 per 1M output tokens)
    cost = (prompt_tokens / 1000000 * 0.03) + (completion_tokens / 1000000 * 2.50)
    
    return {
        "response": response.content,
        "metrics": {
            "response_time": round(end_time - start_time, 2),
            "token_usage": {
                "prompt_tokens": prompt_tokens,
                "completion_tokens": completion_tokens,
                "total_tokens": total_tokens
            },
            "cost": round(cost, 6)
        }
    }

In [64]:
# Dummy data for testing
test_question = "Compare Mohamed Salah and Kevin De Bruyne's performance in the 2022-2023 season."
test_context, baseline_context = get_context(test_question)

RateLimitError: Error code: 429 - {'error': {'message': 'Rate limit exceeded: free-models-per-day. Add 10 credits to unlock 1000 free model requests per day', 'code': 429, 'metadata': {'headers': {'X-RateLimit-Limit': '50', 'X-RateLimit-Remaining': '0', 'X-RateLimit-Reset': '1765843200000'}, 'provider_name': None}}, 'user_id': 'user_31rzWKMjZdo9D93EFSqTxAFhoTn'}

In [60]:
print(test_context)

[{'type': 'structured', 'players': [{'player_name': 'Bukayo Saka', 'player_code': 223340, 'other_props': {'embedding_text_mpnet_v2': [-0.02224762551486492, -0.06235223263502121, -0.006163096055388451, 0.030164822936058044, 0.035846538841724396, -0.018453126773238182, 0.013898775912821293, -0.009416022337973118, -0.004300013650208712, 0.018945889547467232, 0.06940525770187378, -0.008729157969355583, 0.07163375616073608, 0.05596102401614189, 0.030490392819046974, 0.007545436266809702, -0.012045195326209068, -0.04140963777899742, 0.06963994354009628, -0.015607344917953014, -0.022229626774787903, 0.007782161235809326, -0.01975296251475811, 0.017283529043197632, 0.0050915624015033245, -0.01814202591776848, 0.03771252930164337, -0.009174870327115059, -0.023768262937664986, -0.0160784050822258, -0.018165795132517815, -0.05071936547756195, -0.09198794513940811, -0.03813982009887695, 1.7888688716993784e-06, -0.014944811351597309, -0.01669214479625225, -0.03940032795071602, -0.06196806952357292,

In [58]:
print("="*80)
print("=== OpenAI GPT 5.1 ===")
print("="*80)
result = openai_generate(test_context, test_question)
print(f"\nResponse:\n{result['response']}")
print(f"\nMetrics:")
print(f"  Response Time: {result['metrics']['response_time']}s")
print(f"  Tokens - Input: {result['metrics']['token_usage']['prompt_tokens']}, "
      f"Output: {result['metrics']['token_usage']['completion_tokens']}, "
      f"Total: {result['metrics']['token_usage']['total_tokens']}")
print(f"  Cost: ${result['metrics']['cost']}")

=== OpenAI GPT 5.1 ===

Response:
Bukayo Saka is a player for Arsenal, and in the 2023 season he is classified as a midfielder (position: MID).

Metrics:
  Response Time: 3.17s
  Tokens - Input: 12116, Output: 37, Total: 12153
  Cost: $0.015515


In [49]:

print("\n" + "="*80)
print("=== LLama 3.3 (Openrouter) ===")
print("="*80)
result = openrouter_generate(test_context, test_question)
print(f"\nResponse:\n{result['response']}")
print(f"\nMetrics:")
print(f"  Response Time: {result['metrics']['response_time']}s")
print(f"  Tokens - Input: {result['metrics']['token_usage']['prompt_tokens']}, "
      f"Output: {result['metrics']['token_usage']['completion_tokens']}, "
      f"Total: {result['metrics']['token_usage']['total_tokens']}")
print(f"  Cost: ${result['metrics']['cost']}")


=== LLama 3.3 (Openrouter) ===

Response:
The context does not contain enough information to answer the question. There is no data provided about the performance of Mohamed Salah and Erling Haaland in GW15. The context only provides the total points scored by the two players, with Salah scoring 38,221 points and Haaland scoring 41,072 points, but it does not specify the gameweek or any other relevant statistics to determine their performance in GW15.

Metrics:
  Response Time: 4.76s
  Tokens - Input: 508, Output: 103, Total: 611
  Cost: $0.0


In [ ]:

print("\n" + "="*80)
print("=== Gemini 2.5 Flash ===")
print("="*80)
result = gemini_generate(test_context, test_question)
print(f"\nResponse:\n{result['response']}")
print(f"\nMetrics:")
print(f"  Response Time: {result['metrics']['response_time']}s")
print(f"  Tokens - Input: {result['metrics']['token_usage']['prompt_tokens']}, "
      f"Output: {result['metrics']['token_usage']['completion_tokens']}, "
      f"Total: {result['metrics']['token_usage']['total_tokens']}")
print(f"  Cost: ${result['metrics']['cost']}")


=== Gemini 2.5 Flash ===

Response:
The context provided does not contain any information about Mohamed Salah's or Erling Haaland's performance specifically in GW15. It only shows aggregated statistics: Salah (p1) has 38,221 points and Haaland (p2) has 41,072 points in total, but these are not specific to a single gameweek. Therefore, I cannot answer who performed better in GW15.

Metrics:
  Response Time: 2.36s
  Tokens - Input: 387, Output: 294, Total: 681
  Cost: $0.000747


In [ ]:
import streamlit as st



# Set the title of the app
st.title("Graph-RAG Travel Assistant")

# User input for the query
user_query = st.text_input("Ask a question:")

# List of models
models = ['Gemini 2.5 Flash', 'LLama 3.3 (Openrouter)', 'OpenAI GPT 5.1']

# Dropdown to select a model
selected_model = st.selectbox("Select a model", models)

# Show the selected model
st.write(f'You selected: {selected_model}')

# Button to submit the question and process the query
if st.button("Get Answer"):
    # Use the user input and selected model to run the query
    text_context, baseline_context = get_context(user_query)

    # Display the knowledge graph data
    st.subheader("Knowledge Graph Data Retrieved:")
    for i, record in enumerate(baseline_context, start=1):
        st.subheader(f"Result {i}")

        if record["players"]:
            st.markdown("### 🧑 Players")
            st.table(record["players"])

        if record["teams"]:
            st.markdown("### 🏟 Teams")
            st.table(record["teams"])

        if record["positions"]:
            st.markdown("### 📍 Positions")
            st.table(record["positions"])

        if record["fixtures"]:
            st.markdown("### 📅 Fixtures")
            st.table(record["fixtures"])

        if record["relationships"]:
            st.markdown("### 🔗 Relationships")
            st.json(record["relationships"])

        if record["aggregated_stats"]:
            st.markdown("### 📊 Aggregated Stats")
            st.json(record["aggregated_stats"])

    st.divider()

    # Display LLM answer based on the selected model
    st.subheader("LLM Answer:")
    
    if selected_model == 'Gemini 2.5 Flash':
        result = gemini_generate(text_context, user_query)
        print(f"\nResponse:\n{result['response']}")
        st.write(result["response"])
        # Your logic for Model A here
    elif selected_model == 'LLama 3.3 (Openrouter)':
        result = openrouter_generate(text_context, user_query)
        st.write(result["response"])
        # Your logic for Model B here
    else:
        result = openai_generate(text_context, user_query)
        st.write(result["response"])

2025-12-14 05:03:15.865 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-12-14 05:03:16.463 
  command:

    streamlit run /media/sakr/9E9A4F369A4F0A6B/ACL2/fantasy-football-predictor/streamlit-env/lib/python3.12/site-packages/ipykernel_launcher.py [ARGUMENTS]
2025-12-14 05:03:16.463 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-12-14 05:03:16.464 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-12-14 05:03:16.465 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-12-14 05:03:16.466 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-12-14 05:03:16.467 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running 